In [7]:
from pathlib import Path
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

# ========= 配置 =========
F15 = Path("btc_2015.xlsx")
F21 = Path("btc_2021.xlsx")
F25 = Path("btc_2025.xlsx")
OUTDIR = Path("btc_fit_outputs09142025"); OUTDIR.mkdir(exist_ok=True)

# 参考窗口（你要的 24/06 → 25/08/15）
REF_START_25 = pd.Timestamp("2024-06-01")
ANCHOR_25    = pd.Timestamp("2025-08-15")  # 本轮“针尖”假设
REF_END_25   = ANCHOR_25                    # 末端就是锚点
L = (REF_END_25 - REF_START_25).days        # 窗口长度

# ========= 工具 =========
def read_two_col_excel(fp: Path) -> pd.DataFrame:
    """读取两列（时间、价格），兼容“横着两行”格式，去重按时间升序。"""
    assert fp.exists(), f"未找到文件：{fp}"
    df_raw = pd.read_excel(fp, engine="openpyxl", header=None)
    if df_raw.shape[0] <= 3 and df_raw.shape[1] > df_raw.shape[0]:
        df_raw = df_raw.T  # 横排两行 → 竖列
    df = df_raw.iloc[:, :2].copy()
    df.columns = ["ts", "price"]
    df["ts"] = pd.to_datetime(df["ts"], errors="coerce")
    df["price"] = pd.to_numeric(df["price"], errors="coerce")
    df = df.dropna(subset=["ts","price"]).sort_values("ts").drop_duplicates(subset=["ts"], keep="last")
    return df

def to_daily(series_df: pd.DataFrame) -> pd.Series:
    """转为日频收盘(last)并向前填充，拿到连续日索引。"""
    s = series_df.set_index("ts")["price"].resample("D").last()
    s = s.reindex(pd.date_range(s.index.min().normalize(), s.index.max().normalize(), freq="D")).ffill()
    s.index.name = "ts"
    return s

def window_by_end(series: pd.Series, end_date: pd.Timestamp, length_days: int) -> pd.Series:
    """以 end_date 为末端，向前 length_days 天截取窗口；若当日无值，先对齐索引再 ffill。"""
    end_date = pd.Timestamp(end_date).normalize()
    full_idx = pd.date_range(series.index.min(), max(series.index.max(), end_date), freq="D")
    s = series.reindex(full_idx).ffill()
    start_date = max(end_date - pd.Timedelta(days=length_days), s.index.min())
    return s.loc[start_date:end_date]

def pct_curve(series: pd.Series, anchor_date: pd.Timestamp) -> pd.DataFrame:
    """相对锚点百分比曲线（锚点=0%），返回 rel_day / dd_pct 以及锚点价。"""
    anchor_date = pd.Timestamp(anchor_date).normalize()
    anchor_px = float(series.loc[anchor_date] if anchor_date in series.index else series.loc[:anchor_date].iloc[-1])
    rel_day = (series.index - anchor_date).days
    dd_pct = (series / anchor_px - 1.0) * 100.0
    return pd.DataFrame({"rel_day": rel_day, "dd_pct": dd_pct.values}, index=series.index), anchor_px

def fit_metrics(a: pd.Series, b: pd.Series):
    """相关系数 r 与 RMSE（百分点）。a/b 索引需是相同的 rel_day。"""
    idx = a.index.intersection(b.index)
    if len(idx) < 3: return np.nan, np.nan
    A, B = a.loc[idx].values, b.loc[idx].values
    r = float(np.corrcoef(A, B)[0,1]); rmse = float(np.sqrt(np.mean((A-B)**2)))
    return r, rmse

# ========= 读取三表 → 日频 =========
s15 = to_daily(read_two_col_excel(F15))
s21 = to_daily(read_two_col_excel(F21))
s25 = to_daily(read_two_col_excel(F25))

# ========= 2025 参考窗口（24/06 → 25/08/15）=========
win25 = window_by_end(s25, ANCHOR_25, L)
curve25, anchor25 = pct_curve(win25, ANCHOR_25)

# ========= 2021：找 2021 年内最高点 → 回溯同长度 =========
s21_y = s21.loc["2021-01-01":"2021-12-31"]
assert not s21_y.empty, "2021 年数据缺失，请检查 btc_2021.xlsx"
peak21 = s21_y.idxmax()
win21 = window_by_end(s21, peak21, L)
curve21, anchor21 = pct_curve(win21, peak21)

# ========= 2017：找 2017 年内最高点 → 回溯同长度 =========
s17_y = s15.loc["2017-01-01":"2017-12-31"]
assert not s17_y.empty, "2017 年数据缺失，请检查 btc_2015.xlsx 是否覆盖到 2017"
peak17 = s17_y.idxmax()
win17 = window_by_end(s15, peak17, L)
curve17, anchor17 = pct_curve(win17, peak17)

# ========= 统一到 rel_day 轴 =========
c25 = curve25.set_index("rel_day")["dd_pct"]
c21 = curve21.set_index("rel_day")["dd_pct"]
c17 = curve17.set_index("rel_day")["dd_pct"]

# —— 未缩放的拟合程度
r21_raw, rmse21_raw = fit_metrics(c25, c21)
r17_raw, rmse17_raw = fit_metrics(c25, c17)

# ========= “波动退火”：把 2017/2021 的波动缩小到与 2025 窗口一致（仅幅度，不改形状）=========
span = range(-min(90, L), 1)  # 用锚点前最多 90 天估计标准差
std25 = c25.loc[list(filter(lambda x: x in c25.index, span))].std(ddof=0)
std21 = c21.loc[list(filter(lambda x: x in c21.index, span))].std(ddof=0)
std17 = c17.loc[list(filter(lambda x: x in c17.index, span))].std(ddof=0)

scale21 = float(std25 / std21) if (std21 and std21>0) else 1.0
scale17 = float(std25 / std17) if (std17 and std17>0) else 1.0

c21_s = c21 * scale21
c17_s = c17 * scale17

r21_s, rmse21_s = fit_metrics(c25, c21_s)
r17_s, rmse17_s = fit_metrics(c25, c17_s)

# ========= 保存合并数据 =========
merged = (pd.DataFrame({"dd_pct_2025": c25})
          .join(pd.DataFrame({"dd_pct_2021": c21}), how="outer")
          .join(pd.DataFrame({"dd_pct_2017": c17}), how="outer"))
merged["dd_pct_2021_scaled"] = merged["dd_pct_2021"] * scale21
merged["dd_pct_2017_scaled"] = merged["dd_pct_2017"] * scale17
merged.index.name = "rel_day"
merged.to_csv(OUTDIR / "fit_2017_2021_vs_2025_window.csv", encoding="utf-8-sig")

with open(OUTDIR / "fit_2017_2021_vs_2025_metrics.txt","w",encoding="utf-8") as f:
    f.write(f"参考窗口（2025）：{REF_START_25.date()} → {ANCHOR_25.date()}，长度 {L} 天\n")
    f.write(f"锚点价：2017={anchor17:,.2f}，2021={anchor21:,.2f}，2025={anchor25:,.2f}\n\n")
    f.write("—— 未缩放（百分比）——\n")
    f.write(f"2021 vs 2025: r={r21_raw:.4f}, RMSE={rmse21_raw:.3f} pct-pt\n")
    f.write(f"2017 vs 2025: r={r17_raw:.4f}, RMSE={rmse17_raw:.3f} pct-pt\n\n")
    f.write("—— 波动退火（将 2017/2021 的 std 缩放至 2025）——\n")
    f.write(f"scale21={scale21:.3f}, scale17={scale17:.3f}\n")
    f.write(f"2021_scaled vs 2025: r={r21_s:.4f}, RMSE={rmse21_s:.3f} pct-pt\n")
    f.write(f"2017_scaled vs 2025: r={r17_s:.4f}, RMSE={rmse17_s:.3f} pct-pt\n")

# ========= 画图 =========
# 图1：未缩放对比
plt.figure(figsize=(11,4))
plt.plot(c25.index, c25.values, label=f"2025（{REF_START_25.date()}→{ANCHOR_25.date()}）")
plt.plot(c21.index, c21.values, label="2021 对齐窗口（未缩放）")
plt.plot(c17.index, c17.values, label="2017 对齐窗口（未缩放）")
plt.axvline(0, linestyle="--"); plt.xlabel("相对天数（锚点=0）"); plt.ylabel("相对锚点涨跌（%）")
plt.title("锚点归一化百分比：未缩放"); plt.legend(); plt.tight_layout()
plt.savefig(OUTDIR / "fit_unscaled.png", dpi=170); plt.close()

# 图2：波动退火对比
plt.figure(figsize=(11,4))
plt.plot(c25.index, c25.values, label="2025 参考窗口")
plt.plot(c21_s.index, c21_s.values, label=f"2021（缩放 ×{scale21:.2f}）")
plt.plot(c17_s.index, c17_s.values, label=f"2017（缩放 ×{scale17:.2f}）")
plt.axvline(0, linestyle="--"); plt.xlabel("相对天数（锚点=0）"); plt.ylabel("相对锚点涨跌（%）")
plt.title("锚点归一化百分比：波动退火（与 2025 对齐）"); plt.legend(); plt.tight_layout()
plt.savefig(OUTDIR / "fit_scaled.png", dpi=170); plt.close()

print("已输出到：", OUTDIR.resolve())


C:\Users\nickc\AppData\Local\Temp\ipykernel_17908\2252032967.py:26: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["ts"] = pd.to_datetime(df["ts"], errors="coerce")
C:\Users\nickc\AppData\Local\Temp\ipykernel_17908\2252032967.py:26: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["ts"] = pd.to_datetime(df["ts"], errors="coerce")


已输出到： C:\Users\nickc\yanda\btc\btc_fit_outputs09142025


In [8]:
from pathlib import Path
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
# 中文字体&负号 修复（Windows）
from pathlib import Path
from matplotlib import rcParams, font_manager
import matplotlib, shutil
print(matplotlib.get_cachedir())  # 看一下缓存目录
# 然后手动删除该目录下的 fontlist-v***.json（或直接删整个缓存目录）

# 1) 把常见中文字体注册进 Matplotlib
candidates = [
    r"C:\Windows\Fonts\msyh.ttc",     # 微软雅黑
    r"C:\Windows\Fonts\simhei.ttf",   # 黑体
    r"C:\Windows\Fonts\msyhbd.ttc",   # 微软雅黑 Bold
]
for p in candidates:
    if Path(p).exists():
        font_manager.fontManager.addfont(p)

# 2) 设为默认字体 + 修复负号（unicode minus）
rcParams["font.family"] = "sans-serif"
rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "DejaVu Sans"]
rcParams["axes.unicode_minus"] = False  # 避免负号显示成方块

# 可选：如果你把 Noto CJK 放到项目里（跨平台更稳）
# font_manager.fontManager.addfont("fonts/NotoSansCJKsc-Regular.otf")
# rcParams["font.family"] = "Noto Sans CJK SC"

# ========= 配置（按需改） =========
F15 = Path("btc_2015.xlsx")
F21 = Path("btc_2021.xlsx")
F25 = Path("btc_2025.xlsx")
OUTDIR = Path("btc_fit_outputs09142025"); OUTDIR.mkdir(exist_ok=True)

# 参考窗口：用 2024-06-01 → 2025-08-15 作为 2025 的“锚点区间”
REF_START_25 = pd.Timestamp("2024-06-01")
ANCHOR_25    = pd.Timestamp("2025-08-15")   # 2025 假定峰值
POST_DAYS    = 60                           # 峰值后的展示天数（25年会被数据上限截断）
# 2017/2021 的峰值（2017 固定；2021 若想固定也可在此填日期）
PEAK_2017 = pd.Timestamp("2017-12-19")
PEAK_2021 = None   # 比如 pd.Timestamp("2021-11-10")；None=自动取 2021 年内最高日

# —— 波动退火：把旧周期的幅度缩到与 2025 更接近（只改幅度，不改形状）
# 你可以用“手工等级”或“历史标准差”或二者合成（hybrid）
VOL_LEVEL_2017, VOL_LEVEL_2021, VOL_LEVEL_2025 = 15.0, 7.0, 1.5
VOL_ALPHA = 0.5           # =0.5 表示“开方”；=1 表示线性比；可自行调整
SCALE_METHOD = "manual"   # 选 "manual" / "std" / "hybrid"
PRE_STD_SPAN = 90         # 用峰值前多少天估算历史 std（method 含 std/hybrid 时生效）

# ========= 工具 =========
def read_two_col_excel(fp: Path) -> pd.DataFrame:
    """读两列（时间、价格），兼容“横着两行”，去重、按时间升序。"""
    assert fp.exists(), f"未找到文件：{fp}"
    df_raw = pd.read_excel(fp, engine="openpyxl", header=None)
    if df_raw.shape[0] <= 3 and df_raw.shape[1] > df_raw.shape[0]:
        df_raw = df_raw.T
    df = df_raw.iloc[:, :2].copy()
    df.columns = ["ts", "price"]
    df["ts"] = pd.to_datetime(df["ts"], errors="coerce")
    df["price"] = pd.to_numeric(df["price"], errors="coerce")
    df = df.dropna(subset=["ts","price"]).sort_values("ts").drop_duplicates(subset=["ts"], keep="last")
    return df

def to_daily(series_df: pd.DataFrame) -> pd.Series:
    """转日频收盘(last)+前向填充，得到连续日序列。"""
    s = series_df.set_index("ts")["price"].resample("D").last()
    s = s.reindex(pd.date_range(s.index.min().normalize(), s.index.max().normalize(), freq="D")).ffill()
    s.index.name = "ts"
    return s

def window_by_anchor(series: pd.Series, anchor_date: pd.Timestamp, pre_days: int, post_days: int) -> pd.Series:
    """以 anchor 为中心，取前 pre_days、后 post_days 的窗口；若右端超出数据就自动截断。"""
    anchor_date = pd.Timestamp(anchor_date).normalize()
    full_idx = pd.date_range(series.index.min(), max(series.index.max(), anchor_date), freq="D")
    s = series.reindex(full_idx).ffill()
    lo = max(anchor_date - pd.Timedelta(days=pre_days), s.index.min())
    hi = min(anchor_date + pd.Timedelta(days=post_days), s.index.max())
    return s.loc[lo:hi]

def pct_curve(series: pd.Series, anchor_date: pd.Timestamp) -> pd.DataFrame:
    """相对锚点百分比曲线（锚点=0%），index=日期，含 rel_day & dd_pct。"""
    anchor_date = pd.Timestamp(anchor_date).normalize()
    anchor_px = float(series.loc[anchor_date] if anchor_date in series.index else series.loc[:anchor_date].iloc[-1])
    rel_day = (series.index - anchor_date).days
    dd_pct = (series / anchor_px - 1.0) * 100.0
    return pd.DataFrame({"rel_day": rel_day, "dd_pct": dd_pct.values}, index=series.index), anchor_px

def fit_metrics(a: pd.Series, b: pd.Series):
    idx = a.index.intersection(b.index)
    if len(idx) < 3: return np.nan, np.nan
    A, B = a.loc[idx].values, b.loc[idx].values
    r = float(np.corrcoef(A, B)[0,1]); rmse = float(np.sqrt(np.mean((A-B)**2)))
    return r, rmse

def scale_factor(std_old, std_new, level_old, level_new, method="manual", alpha=0.5):
    """
    计算缩放系数（乘到旧周期上）：
      - manual : (level_new / level_old) ** alpha     # 你给的“9→3→1 等级”，alpha=0.5=开方
      - std    : std_new / std_old                    # 用历史标准差比例
      - hybrid : (std_new / std_old) * (level_new / level_old) ** alpha
    """
    if method == "manual":
        return (level_new / level_old) ** alpha
    elif method == "std":
        return (std_new / std_old) if std_old and std_old > 0 else 1.0
    else:  # hybrid
        left  = (std_new / std_old) if std_old and std_old > 0 else 1.0
        right = (level_new / level_old) ** alpha
        return left * right

# ========= 读三表 → 日频 =========
s15 = to_daily(read_two_col_excel(F15))
s21 = to_daily(read_two_col_excel(F21))
s25 = to_daily(read_two_col_excel(F25))

# ========= 2025 参考窗（含峰后 POST_DAYS）=========
pre_len = (ANCHOR_25 - REF_START_25).days
win25 = window_by_anchor(s25, ANCHOR_25, pre_len, POST_DAYS)
curve25, anchor25 = pct_curve(win25, ANCHOR_25)
c25 = curve25.set_index("rel_day")["dd_pct"]

# ========= 2021：峰值（自动或固定）→ 同长度窗口（含峰后）=========
s21_y = s21.loc["2021-01-01":"2021-12-31"]
assert not s21_y.empty, "2021 年数据缺失，请检查 btc_2021.xlsx"
peak21 = (PEAK_2021 if PEAK_2021 is not None else s21_y.idxmax())
win21 = window_by_anchor(s21, peak21, pre_len, POST_DAYS)
curve21, anchor21 = pct_curve(win21, peak21)
c21 = curve21.set_index("rel_day")["dd_pct"]

# ========= 2017：峰值固定 11/30 → 同长度窗口（含峰后）=========
s17_y = s15.loc["2017-01-01":"2017-12-31"]
assert not s17_y.empty, "2017 年数据缺失，请检查 btc_2015.xlsx 是否覆盖到 2017"
peak17 = PEAK_2017
win17 = window_by_anchor(s15, peak17, pre_len, POST_DAYS)
curve17, anchor17 = pct_curve(win17, peak17)
c17 = curve17.set_index("rel_day")["dd_pct"]

# ========= 预估 std（用于 std/hybrid）——用峰前 PRE_STD_SPAN 天 =========
def pre_std(c):
    idx = [d for d in range(-min(PRE_STD_SPAN, pre_len), 1) if d in c.index]
    return c.loc[idx].std(ddof=0)

std25 = pre_std(c25); std21 = pre_std(c21); std17 = pre_std(c17)

# ========= 计算缩放系数（把 2017/2021 缩成接近 2025）=========
scale21 = scale_factor(std21, std25, VOL_LEVEL_2021, VOL_LEVEL_2025, method=SCALE_METHOD, alpha=VOL_ALPHA)
scale17 = scale_factor(std17, std25, VOL_LEVEL_2017, VOL_LEVEL_2025, method=SCALE_METHOD, alpha=VOL_ALPHA)
c21_s, c17_s = c21 * scale21, c17 * scale17

# ========= 拟合指标（未缩放 & 缩放后；峰前/峰后 分开看）=========
def split_metrics(c_old, name):
    pre_idx  = [i for i in c25.index if i <= 0 and i in c_old.index]
    post_idx = [i for i in c25.index if i >= 0 and i in c_old.index]
    r_pre, rmse_pre   = fit_metrics(c25.loc[pre_idx],  c_old.loc[pre_idx])
    r_post, rmse_post = fit_metrics(c25.loc[post_idx], c_old.loc[post_idx])
    return r_pre, rmse_pre, r_post, rmse_post

m21_raw  = split_metrics(c21,  "2021_raw")
m17_raw  = split_metrics(c17,  "2017_raw")
m21_scal = split_metrics(c21_s,"2021_scaled")
m17_scal = split_metrics(c17_s,"2017_scaled")

# ========= 保存合并数据 =========
merged = (pd.DataFrame({"dd_pct_2025": c25})
          .join(pd.DataFrame({"dd_pct_2021": c21}), how="outer")
          .join(pd.DataFrame({"dd_pct_2017": c17}), how="outer"))
merged["dd_pct_2021_scaled"] = merged["dd_pct_2021"] * scale21
merged["dd_pct_2017_scaled"] = merged["dd_pct_2017"] * scale17
merged.index.name = "rel_day"
merged.to_csv(OUTDIR / "fit_2017_2021_vs_2025_window_with_tail.csv", encoding="utf-8-sig")

with open(OUTDIR / "fit_metrics_with_tail.txt","w",encoding="utf-8") as f:
    f.write(f"窗口（2025）：{REF_START_25.date()} → {ANCHOR_25.date()}，峰后展示 {POST_DAYS} 天\n")
    f.write(f"峰值：2017={peak17.date()}，2021={peak21.date()}，2025={ANCHOR_25.date()}\n")
    f.write(f"锚点价：2017={anchor17:,.2f}，2021={anchor21:,.2f}，2025={anchor25:,.2f}\n")
    f.write(f"std(峰前)：2017={std17:.3f}，2021={std21:.3f}，2025={std25:.3f}\n")
    f.write(f"scale(method={SCALE_METHOD}, alpha={VOL_ALPHA}): 2017→25 = {scale17:.3f}，2021→25 = {scale21:.3f}\n\n")
    f.write("—— 未缩放 ——\n")
    f.write(f"2021: r_pre={m21_raw[0]:.4f}, rmse_pre={m21_raw[1]:.3f} | r_post={m21_raw[2]:.4f}, rmse_post={m21_raw[3]:.3f}\n")
    f.write(f"2017: r_pre={m17_raw[0]:.4f}, rmse_pre={m17_raw[1]:.3f} | r_post={m17_raw[2]:.4f}, rmse_post={m17_raw[3]:.3f}\n\n")
    f.write("—— 缩放后（波动退火）——\n")
    f.write(f"2021_scaled: r_pre={m21_scal[0]:.4f}, rmse_pre={m21_scal[1]:.3f} | r_post={m21_scal[2]:.4f}, rmse_post={m21_scal[3]:.3f}\n")
    f.write(f"2017_scaled: r_pre={m17_scal[0]:.4f}, rmse_pre={m17_scal[1]:.3f} | r_post={m17_scal[2]:.4f}, rmse_post={m17_scal[3]:.3f}\n")

# ========= 画图（保存）=========
# 图1：波动退火（与你发的图类似，但含峰后）
plt.figure(figsize=(12,4.5))
plt.plot(c25.index, c25.values, label="2025 参考窗")
plt.plot(c21_s.index, c21_s.values, label=f"2021（缩放 ×{scale21:.2f}）")
plt.plot(c17_s.index, c17_s.values, label=f"2017（缩放 ×{scale17:.2f}）")
plt.axvline(0, linestyle="--")
plt.xlabel("相对天数（锚点=0）"); plt.ylabel("相对锚点涨跌（%）")
plt.title("锚点归一化百分比：波动退火后，与 2025 对齐（含峰后）")
plt.legend(); plt.tight_layout()
plt.savefig(OUTDIR / "fit_scaled_with_tail.png", dpi=170); plt.close()

# 图2：未缩放，对比真实幅度（方便直观看到 2017≫2021≫2025 的波动差）
plt.figure(figsize=(12,4.5))
plt.plot(c25.index, c25.values, label="2025（原幅度）")
plt.plot(c21.index, c21.values, label="2021（原幅度）")
plt.plot(c17.index, c17.values, label="2017（原幅度）")
plt.axvline(0, linestyle="--")
plt.xlabel("相对天数（锚点=0）"); plt.ylabel("相对锚点涨跌（%）")
plt.title("锚点归一化百分比：原始幅度（含峰后）")
plt.legend(); plt.tight_layout()
plt.savefig(OUTDIR / "fit_unscaled_with_tail0588v2.png", dpi=170); plt.close()

print("已输出到：", OUTDIR.resolve())



C:\Users\nickc\.matplotlib


C:\Users\nickc\AppData\Local\Temp\ipykernel_17908\1257596865.py:60: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["ts"] = pd.to_datetime(df["ts"], errors="coerce")
C:\Users\nickc\AppData\Local\Temp\ipykernel_17908\1257596865.py:60: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["ts"] = pd.to_datetime(df["ts"], errors="coerce")


已输出到： C:\Users\nickc\yanda\btc\btc_fit_outputs09142025


In [9]:
from pathlib import Path
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

# ========= 配置 =========
F15 = Path("btc_2015.xlsx")
F21 = Path("btc_2021.xlsx")
F25 = Path("btc_2025.xlsx")
OUTDIR = Path("btc_fit_outputs09142025"); OUTDIR.mkdir(exist_ok=True)

# 减半 & 峰值（可按你口径改）
HALVING_2016 = pd.Timestamp("2016-07-09")
HALVING_2020 = pd.Timestamp("2020-05-11")
HALVING_2024 = pd.Timestamp("2024-04-20")

PEAK_2017 = pd.Timestamp("2017-12-19")     # 你指定
PEAK_2021 = None                           # None=自动找 2021 年最高点；或填固定日期
PEAK_2025 = pd.Timestamp("2025-08-15")     # 假设峰值
POST_DAYS  = 60                            # 峰后展示天数
MAX_PRE_DAYS = 300                         # 峰前最多展示天数（None=不截断）

# —— 波动退火（幅度对齐到 2025，不改形状）——
VOL_LEVEL_2017, VOL_LEVEL_2021, VOL_LEVEL_2025 = 9.0, 3.0, 1.0
VOL_ALPHA   = 0.5                 # =0.5≈“开方”关系；=1 线性
SCALE_METHOD = "manual"           # "manual" / "std" / "hybrid"
PRE_STD_SPAN = 90                 # 用峰前多少天估算 std（std/hybrid 时生效）

# （可选）中文与负号
from matplotlib import rcParams, font_manager
for p in [r"C:\Windows\Fonts\msyh.ttc", r"C:\Windows\Fonts\simhei.ttf"]:
    if Path(p).exists(): font_manager.fontManager.addfont(p)
rcParams["font.family"] = "sans-serif"
rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "DejaVu Sans"]
rcParams["axes.unicode_minus"] = False

# ========= 工具 =========
def read_two_col_excel(fp: Path) -> pd.DataFrame:
    assert fp.exists(), f"未找到文件：{fp}"
    df = pd.read_excel(fp, engine="openpyxl", header=None)
    if df.shape[0] <= 3 and df.shape[1] > df.shape[0]:
        df = df.T
    df = df.iloc[:, :2].copy()
    df.columns = ["ts","price"]
    df["ts"] = pd.to_datetime(df["ts"], errors="coerce")
    df["price"] = pd.to_numeric(df["price"], errors="coerce")
    return (df.dropna().drop_duplicates("ts").sort_values("ts"))

def to_daily(df: pd.DataFrame) -> pd.Series:
    s = df.set_index("ts")["price"].resample("D").last()
    s = s.reindex(pd.date_range(s.index.min().normalize(), s.index.max().normalize(), freq="D")).ffill()
    s.index.name = "ts";  return s

def window_halving_to_peak(series: pd.Series, halving: pd.Timestamp, peak: pd.Timestamp, post_days=0, max_pre=None):
    """从减半(halving)到峰值(peak) + 峰后 post_days；若 max_pre 给定，则仅取峰前 max_pre 天"""
    halving = pd.Timestamp(halving).normalize()
    peak    = pd.Timestamp(peak   ).normalize()
    full_idx = pd.date_range(series.index.min(), max(series.index.max(), peak), freq="D")
    s = series.reindex(full_idx).ffill()
    start = halving
    if max_pre is not None:
        start = max(peak - pd.Timedelta(days=max_pre), start)
    end = min(peak + pd.Timedelta(days=post_days), s.index.max())
    return s.loc[start:end]

def pct_curve(series: pd.Series, anchor_date: pd.Timestamp):
    anchor_date = pd.Timestamp(anchor_date).normalize()
    anchor_px = float(series.loc[anchor_date] if anchor_date in series.index else series.loc[:anchor_date].iloc[-1])
    rel_day = (series.index - anchor_date).days
    dd_pct  = (series / anchor_px - 1.0) * 100.0
    return pd.DataFrame({"rel_day": rel_day, "dd_pct": dd_pct.values}, index=series.index), anchor_px

def pre_std(curve_rel_day_series: pd.Series, span=90):
    idx = [d for d in range(-span, 1) if d in curve_rel_day_series.index]
    return curve_rel_day_series.loc[idx].std(ddof=0)

def scale_factor(std_old, std_new, level_old, level_new, method="manual", alpha=0.5):
    if method=="manual": return (level_new/level_old)**alpha
    if method=="std":    return (std_new/std_old) if std_old and std_old>0 else 1.0
    # hybrid
    left  = (std_new/std_old) if std_old and std_old>0 else 1.0
    right = (level_new/level_old)**alpha
    return left*right

def fit_metrics(a: pd.Series, b: pd.Series):
    idx = a.index.intersection(b.index)
    if len(idx)<3: return np.nan, np.nan
    A, B = a.loc[idx].values, b.loc[idx].values
    r = float(np.corrcoef(A,B)[0,1]); rmse = float(np.sqrt(np.mean((A-B)**2)))
    return r, rmse

# ========= 读取为日频 =========
s15, s21, s25 = map(to_daily, [read_two_col_excel(F15), read_two_col_excel(F21), read_two_col_excel(F25)])

# ========= 构造三段：减半→峰值（含峰后）=========
# 2017（上一轮）
win17 = window_halving_to_peak(s15, HALVING_2016, PEAK_2017, POST_DAYS, MAX_PRE_DAYS)
curve17, anchor17 = pct_curve(win17, PEAK_2017); c17 = curve17.set_index("rel_day")["dd_pct"]
# 2021（上一轮）
s21_year = s21.loc["2021-01-01":"2021-12-31"]
peak21 = (PEAK_2021 if PEAK_2021 is not None else s21_year.idxmax())
win21 = window_halving_to_peak(s21, HALVING_2020, peak21, POST_DAYS, MAX_PRE_DAYS)
curve21, anchor21 = pct_curve(win21, peak21); c21 = curve21.set_index("rel_day")["dd_pct"]
# 2025（本轮）
win25 = window_halving_to_peak(s25, HALVING_2024, PEAK_2025, POST_DAYS, MAX_PRE_DAYS)
curve25, anchor25 = pct_curve(win25, PEAK_2025); c25 = curve25.set_index("rel_day")["dd_pct"]

# 统计：减半→峰值天数
d_17 = (PEAK_2017 - HALVING_2016).days
d_21 = (peak21    - HALVING_2020).days
d_25 = (PEAK_2025 - HALVING_2024).days

# ========= 退火缩放到 2025 的幅度 =========
std25 = pre_std(c25, span=min(PRE_STD_SPAN, max(5, d_25)))
std21 = pre_std(c21, span=min(PRE_STD_SPAN, max(5, d_21)))
std17 = pre_std(c17, span=min(PRE_STD_SPAN, max(5, d_17)))
scale21 = scale_factor(std21, std25, VOL_LEVEL_2021, VOL_LEVEL_2025, method=SCALE_METHOD, alpha=VOL_ALPHA)
scale17 = scale_factor(std17, std25, VOL_LEVEL_2017, VOL_LEVEL_2025, method=SCALE_METHOD, alpha=VOL_ALPHA)
c21_s, c17_s = c21*scale21, c17*scale17

# ========= 拟合指标（峰前/峰后各算一次）=========
def split_metrics(c_old):
    pre_idx  = [i for i in c25.index if i<=0 and i in c_old.index]
    post_idx = [i for i in c25.index if i>=0 and i in c_old.index]
    return (fit_metrics(c25.loc[pre_idx], c_old.loc[pre_idx]) +
            fit_metrics(c25.loc[post_idx], c_old.loc[post_idx]))

m21_raw  = split_metrics(c21)
m17_raw  = split_metrics(c17)
m21_scal = split_metrics(c21_s)
m17_scal = split_metrics(c17_s)

# ========= 保存数据与指标 =========
merged = (pd.DataFrame({"dd_pct_2025": c25})
          .join(pd.DataFrame({"dd_pct_2021": c21}), how="outer")
          .join(pd.DataFrame({"dd_pct_2017": c17}), how="outer"))
merged["dd_pct_2021_scaled"] = merged["dd_pct_2021"] * scale21
merged["dd_pct_2017_scaled"] = merged["dd_pct_2017"] * scale17
merged.index.name = "rel_day"
merged.to_csv(OUTDIR/"halving_to_peak_aligned0830.csv", encoding="utf-8-sig")

with open(OUTDIR/"halving_to_peak_metrics0830.txt","w",encoding="utf-8") as f:
    f.write(f"减半→峰值天数：2017={d_17}，2021={d_21}，2025(假设)={d_25}\n")
    f.write(f"std(峰前)：2017={std17:.3f}，2021={std21:.3f}，2025={std25:.3f}\n")
    f.write(f"scale(method={SCALE_METHOD}, alpha={VOL_ALPHA}): 2017→25={scale17:.3f}，2021→25={scale21:.3f}\n\n")
    f.write("未缩放 r/RMSE（峰前 | 峰后）：\n")
    f.write(f"2021_raw : pre r={m21_raw[0]:.4f}, rmse={m21_raw[1]:.3f} | post r={m21_raw[2]:.4f}, rmse={m21_raw[3]:.3f}\n")
    f.write(f"2017_raw : pre r={m17_raw[0]:.4f}, rmse={m17_raw[1]:.3f} | post r={m17_raw[2]:.4f}, rmse={m17_raw[3]:.3f}\n\n")
    f.write("退火后 r/RMSE（峰前 | 峰后）：\n")
    f.write(f"2021_scal: pre r={m21_scal[0]:.4f}, rmse={m21_scal[1]:.3f} | post r={m21_scal[2]:.4f}, rmse={m21_scal[3]:.3f}\n")
    f.write(f"2017_scal: pre r={m17_scal[0]:.4f}, rmse={m17_scal[1]:.3f} | post r={m17_scal[2]:.4f}, rmse={m17_scal[3]:.3f}\n")

# ========= 画图 =========
# 图1：退火（缩放）后，从减半到峰值+峰后
plt.figure(figsize=(12.5,4.5))
plt.plot(c25.index, c25.values, label="2025（参考）")
plt.plot(c21_s.index, c21_s.values, label=f"2021（缩放 ×{scale21:.2f}）")
plt.plot(c17_s.index, c17_s.values, label=f"2017（缩放 ×{scale17:.2f}）")
plt.axvline(0, linestyle="--")
plt.xlabel("相对天数（锚点=0）"); plt.ylabel("相对锚点涨跌（%）")
plt.title("减半→峰值对齐（含峰后）：波动退火后")
plt.legend(); plt.tight_layout()
plt.savefig(OUTDIR/"halving_to_peak_scaled0830.png", dpi=170); plt.close()

# 图2：未缩放原始幅度
plt.figure(figsize=(12.5,4.5))
plt.plot(c25.index, c25.values, label="2025（原幅度）")
plt.plot(c21.index, c21.values, label="2021（原幅度）")
plt.plot(c17.index, c17.values, label="2017（原幅度）")
plt.axvline(0, linestyle="--")
plt.xlabel("相对天数（锚点=0）"); plt.ylabel("相对锚点涨跌（%）")
plt.title("减半→峰值对齐（含峰后）：原始幅度")
plt.legend(); plt.tight_layout()
plt.savefig(OUTDIR/"halving_to_peak_unscaled0830.png", dpi=170); plt.close()

print("已输出到：", OUTDIR.resolve())


C:\Users\nickc\AppData\Local\Temp\ipykernel_17908\4280444751.py:44: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["ts"] = pd.to_datetime(df["ts"], errors="coerce")
C:\Users\nickc\AppData\Local\Temp\ipykernel_17908\4280444751.py:44: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["ts"] = pd.to_datetime(df["ts"], errors="coerce")


已输出到： C:\Users\nickc\yanda\btc\btc_fit_outputs09142025


In [4]:
# ========= 只改这一行 =========
COMPARE_YEARS = [2021, 2025]   # 例如 [2017,2021,2025] / [2021,2025] / [2017,2021]

# ========= 下面都不用改（除非你想微调）=========
from pathlib import Path
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

OUTDIR = Path("btc_fit_outputs"); OUTDIR.mkdir(exist_ok=True)

DATA_FILES = {2017: Path("btc_2015.xlsx"), 2021: Path("btc_2021.xlsx"), 2025: Path("btc_2025.xlsx")}
HALVINGS   = {2017: pd.Timestamp("2016-07-09"), 2021: pd.Timestamp("2020-05-11"), 2025: pd.Timestamp("2024-04-20")}
PEAKS_DEF  = {2017: pd.Timestamp("2017-12-18"), 2021: None,                    2025: pd.Timestamp("2025-08-15")}

POST_DAYS     = 60      # 峰值后的展示天数
MAX_PRE_DAYS  = 300     # 峰值前最多天数（None = 不截断）

# 波动“退火”到参考年（只改幅度，不改形状）
VOL_LEVELS   = {2017: 9.0, 2021: 3.0, 2025: 1.0}
VOL_ALPHA    = 0.5                 # 0.5≈“开方”；1.0=线性
SCALE_METHOD = "manual"            # "manual" / "std" / "hybrid"
PRE_STD_SPAN = 90                  # std/hybrid 时峰前取多少天

# 中文字体（Windows 可用）
from matplotlib import rcParams, font_manager
for p in [r"C:\Windows\Fonts\msyh.ttc", r"C:\Windows\Fonts\simhei.ttf"]:
    if Path(p).exists(): font_manager.fontManager.addfont(p)
rcParams["font.family"] = "sans-serif"
rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "DejaVu Sans"]
rcParams["axes.unicode_minus"] = False

# ============ 工具函数 ============
def read_two_col_excel(fp: Path) -> pd.DataFrame:
    assert fp.exists(), f"未找到文件：{fp}"
    df = pd.read_excel(fp, engine="openpyxl", header=None)
    if df.shape[0] <= 3 and df.shape[1] > df.shape[0]: df = df.T
    df = df.iloc[:, :2].copy()
    df.columns = ["ts","price"]
    df["ts"] = pd.to_datetime(df["ts"], errors="coerce")
    df["price"] = pd.to_numeric(df["price"], errors="coerce")
    return df.dropna().drop_duplicates("ts").sort_values("ts")

def to_daily(df: pd.DataFrame) -> pd.Series:
    s = df.set_index("ts")["price"].resample("D").last()
    s = s.reindex(pd.date_range(s.index.min().normalize(), s.index.max().normalize(), freq="D")).ffill()
    s.index.name = "ts"; return s

def window_halving_to_peak(series: pd.Series, halving: pd.Timestamp, peak: pd.Timestamp,
                           post_days=0, max_pre=None) -> pd.Series:
    halving, peak = pd.Timestamp(halving).normalize(), pd.Timestamp(peak).normalize()
    full_idx = pd.date_range(series.index.min(), max(series.index.max(), peak), freq="D")
    s = series.reindex(full_idx).ffill()
    start = halving
    if max_pre is not None:
        start = max(peak - pd.Timedelta(days=max_pre), start)
    end = min(peak + pd.Timedelta(days=post_days), s.index.max())
    return s.loc[start:end]

def pct_curve(series: pd.Series, anchor_date: pd.Timestamp):
    anchor_date = pd.Timestamp(anchor_date).normalize()
    anchor_px = float(series.loc[anchor_date] if anchor_date in series.index else series.loc[:anchor_date].iloc[-1])
    rel_day = (series.index - anchor_date).days
    dd_pct  = (series / anchor_px - 1.0) * 100.0
    return pd.DataFrame({"rel_day": rel_day, "dd_pct": dd_pct.values}, index=series.index), anchor_px

def pre_std(rel_series: pd.Series, span=90):
    idx = [d for d in range(-span, 1) if d in rel_series.index]
    return rel_series.loc[idx].std(ddof=0)

def scale_factor(std_old, std_new, level_old, level_new, method="manual", alpha=0.5):
    if method == "manual": return (level_new / level_old) ** alpha
    if method == "std":    return (std_new / std_old) if std_old and std_old > 0 else 1.0
    left  = (std_new / std_old) if std_old and std_old > 0 else 1.0
    right = (level_new / level_old) ** alpha
    return left * right

def fit_metrics(a: pd.Series, b: pd.Series):
    idx = a.index.intersection(b.index)
    if len(idx) < 3: return np.nan, np.nan
    A, B = a.loc[idx].values, b.loc[idx].values
    r = float(np.corrcoef(A,B)[0,1]); rmse = float(np.sqrt(np.mean((A-B)**2)))
    return r, rmse

# ============ 主流程 ============
YEARS = list(sorted(set(COMPARE_YEARS)))
REF_YEAR = YEARS[-1]  # 参考年=最后一个
assert all(y in DATA_FILES for y in YEARS), f"未知年份：{[y for y in YEARS if y not in DATA_FILES]}"

# 读取为日频
daily = {y: to_daily(read_two_col_excel(DATA_FILES[y])) for y in YEARS}

# 峰值：默认表（2021=None 则自动找 2021 年内最高日）
peaks = dict(PEAKS_DEF)
if 2021 in YEARS and peaks[2021] is None:
    y21 = daily[2021].loc["2021-01-01":"2021-12-31"]; assert not y21.empty, "2021 年数据缺失"
    peaks[2021] = y21.idxmax()

# 从减半→峰值 + 峰后，转为“相对锚点百分比”
curves, anchors = {}, {}
for y in YEARS:
    w = window_halving_to_peak(daily[y], HALVINGS[y], peaks[y], POST_DAYS, MAX_PRE_DAYS)
    cdf, anch = pct_curve(w, peaks[y])
    curves[y]  = cdf.set_index("rel_day")["dd_pct"]
    anchors[y] = anch

# 峰前 std（用于 std/hybrid）
stds = {y: pre_std(curves[y], span=min(PRE_STD_SPAN, MAX_PRE_DAYS or PRE_STD_SPAN)) for y in YEARS}

# 退火：把其它年份幅度缩到参考年
cref, std_ref, lvl_ref = curves[REF_YEAR], stds[REF_YEAR], VOL_LEVELS[REF_YEAR]
scales = {}
curves_scaled = {REF_YEAR: cref}
for y in YEARS:
    if y == REF_YEAR: scales[y] = 1.0; continue
    scales[y] = scale_factor(stds[y], std_ref, VOL_LEVELS[y], lvl_ref,
                             method=SCALE_METHOD, alpha=VOL_ALPHA)
    curves_scaled[y] = curves[y] * scales[y]

# 指标（相对参考年；峰前/峰后各算一次）
def split_metrics(c_old):
    pre_idx  = [i for i in cref.index if i <= 0 and i in c_old.index]
    post_idx = [i for i in cref.index if i >= 0 and i in c_old.index]
    return (fit_metrics(cref.loc[pre_idx],  c_old.loc[pre_idx]) +
            fit_metrics(cref.loc[post_idx], c_old.loc[post_idx]))

metrics_raw, metrics_scal = {}, {}
for y in YEARS:
    if y == REF_YEAR: continue
    metrics_raw[y]  = split_metrics(curves[y])
    metrics_scal[y] = split_metrics(curves_scaled[y])

# 保存 CSV & 指标
tag = "_".join(map(str, YEARS))
merged = pd.DataFrame({f"dd_pct_{y}": curves[y] for y in YEARS})
for y in YEARS: merged[f"dd_pct_{y}_scaled"] = curves_scaled[y]
merged.index.name = "rel_day"
merged.to_csv(OUTDIR / f"compare_{tag}_halving_to_peak.csv", encoding="utf-8-sig")

with open(OUTDIR / f"compare_{tag}_metrics.txt","w",encoding="utf-8") as f:
    f.write(f"参考年 ref = {REF_YEAR}\n峰值：\n")
    for y in YEARS: f.write(f"  {y}: {pd.Timestamp(peaks[y]).date()}  锚点价={anchors[y]:,.2f}\n")
    f.write("\n峰前 std：\n"); 
    for y in YEARS: f.write(f"  {y}: {stds[y]:.3f}\n")
    f.write(f"\n缩放方法: {SCALE_METHOD}, VOL_ALPHA={VOL_ALPHA}\n缩放系数：\n")
    for y in YEARS: f.write(f"  {y}: ×{scales[y]:.3f}\n")
    f.write("\n—— 与参考年相比（未缩放 | 缩放后），r 与 RMSE（峰前 | 峰后）——\n")
    for y in YEARS:
        if y == REF_YEAR: continue
        r0,e0,r1,e1 = metrics_raw[y]
        R0,E0,R1,E1 = metrics_scal[y]
        f.write(f"{y} vs {REF_YEAR}  未缩放: pre r={r0:.4f}, rmse={e0:.3f} | post r={r1:.4f}, rmse={e1:.3f}\n")
        f.write(f"{y} vs {REF_YEAR}  缩放后: pre r={R0:.4f}, rmse={E0:.3f} | post r={R1:.4f}, rmse={E1:.3f}\n")

# 画图：缩放后
plt.figure(figsize=(12.5,4.5))
for y in YEARS:
    label = f"{y}（参考）" if y==REF_YEAR else f"{y}（缩放 ×{scales[y]:.2f}）"
    plt.plot(curves_scaled[y].index, curves_scaled[y].values, label=label)
plt.axvline(0, linestyle="--"); plt.xlabel("相对天数（锚点=0）"); plt.ylabel("相对锚点涨跌（%）")
plt.title("减半→峰值对齐（含峰后）：波动退火后"); plt.legend(); plt.tight_layout()
plt.savefig(OUTDIR / f"compare_{tag}_scaled.png", dpi=170); plt.close()

# 画图：未缩放
plt.figure(figsize=(12.5,4.5))
for y in YEARS: plt.plot(curves[y].index, curves[y].values, label=f"{y}（原幅度）")
plt.axvline(0, linestyle="--"); plt.xlabel("相对天数（锚点=0）"); plt.ylabel("相对锚点涨跌（%）")
plt.title("减半→峰值对齐（含峰后）：原始幅度"); plt.legend(); plt.tight_layout()
plt.savefig(OUTDIR / f"compare_{tag}_unscaled.png", dpi=170); plt.close()

print("输出目录：", OUTDIR.resolve())
print("CSV：", (OUTDIR / f"compare_{tag}_halving_to_peak.csv").resolve())
print("指标：", (OUTDIR / f"compare_{tag}_metrics.txt").resolve())


C:\Users\nickc\AppData\Local\Temp\ipykernel_17908\1882606506.py:39: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["ts"] = pd.to_datetime(df["ts"], errors="coerce")


输出目录： C:\Users\nickc\yanda\btc\btc_fit_outputs
CSV： C:\Users\nickc\yanda\btc\btc_fit_outputs\compare_2021_2025_halving_to_peak.csv
指标： C:\Users\nickc\yanda\btc\btc_fit_outputs\compare_2021_2025_metrics.txt


In [5]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ============ 配置（可按需改）============
IN_XLSX = Path("btc_fit_outputs/ahr999_daily.xlsx")
IN_CSV  = Path("btc_fit_outputs/ahr999_daily.csv")
OUTDIR  = Path("btc_fit_outputs"); OUTDIR.mkdir(exist_ok=True)

# 三次减半（近三轮）
HALVINGS = {
    2017: pd.Timestamp("2016-07-09"),
    2021: pd.Timestamp("2020-05-11"),
    2025: pd.Timestamp("2024-04-20"),
}

# 三次峰值（2017 修正为 12-18；2021 常用 11-10；2025 为你的假设 08-15）
PEAKS = {
    2017: pd.Timestamp("2017-12-18"),
    2021: pd.Timestamp("2021-11-10"),
    2025: pd.Timestamp("2025-08-15"),
}

PEAK_WIN_DAYS = 7   # 峰值±7天窗口内取 AHR999 最大值作为“窗口峰”
PLOT_FILENAME = OUTDIR / "ahr999_decay_fit.png"
XLSX_OUT      = OUTDIR / "ahr999_decay_analysis.xlsx"

# ============ 读取 AHR999 日线 ============
def load_ahr999(xlsx_path: Path, csv_path: Path) -> pd.DataFrame:
    if xlsx_path.exists():
        df = pd.read_excel(xlsx_path, sheet_name="AHR999")
    elif csv_path.exists():
        df = pd.read_csv(csv_path)
    else:
        raise FileNotFoundError("未找到 ahr999_daily.xlsx / ahr999_daily.csv，请先运行前面的 AHR999 生成脚本。")
    # 解析日期索引
    date_col = df.columns[0]
    df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
    df = df.dropna(subset=[date_col]).set_index(date_col).sort_index()
    # 关键列校验
    need_cols = {"ahr999", "price", "gma200", "estimate_price", "avg_index", "estimate_index"}
    miss = [c for c in need_cols if c not in df.columns]
    if miss:
        raise ValueError(f"输入缺少列：{miss}；请确认用的是我提供的 AHR999 导出脚本。")
    return df

ahr = load_ahr999(IN_XLSX, IN_CSV)

# ============ 按“减半→峰值”切片并抽取指标 ============
records = []
for yr in [2017, 2021, 2025]:
    halving = HALVINGS[yr]
    peak    = PEAKS[yr]
    # 为防止日期不在索引上，做对齐
    def nearest(ts):
        # 最近一天（向前优先）
        if ts in ahr.index: return ts
        before = ahr.index[ahr.index <= ts]
        after  = ahr.index[ahr.index >= ts]
        if len(before)>0: return before.max()
        return after.min()
    halving = nearest(halving)
    peak    = nearest(peak)
    seg = ahr.loc[halving:peak].copy()
    if seg.empty:
        raise ValueError(f"{yr} 区间空：{halving} → {peak}，请检查源数据覆盖。")

    ahr_at_peak = float(ahr.loc[peak, "ahr999"])

    lo, hi = peak - pd.Timedelta(days=PEAK_WIN_DAYS), peak + pd.Timedelta(days=PEAK_WIN_DAYS)
    win = ahr.loc[lo:hi, "ahr999"]
    ahr_peak_win = float(win.max())
    ahr_peak_win_date = win.idxmax()

    pre_median = float(seg["ahr999"].median())
    pre_p90    = float(seg["ahr999"].quantile(0.90))
    pre_max    = float(seg["ahr999"].max())
    days_halving_to_peak = int((peak - halving).days)

    records.append({
        "cycle_peak_year": yr,
        "halving_date": halving.date(),
        "peak_date": peak.date(),
        "days_halving_to_peak": days_halving_to_peak,
        "ahr_at_peak_date": ahr_at_peak,
        "ahr_peak_window_max": ahr_peak_win,
        "ahr_peak_window_max_date": ahr_peak_win_date.date(),
        "ahr_pre_median": pre_median,
        "ahr_pre_p90": pre_p90,
        "ahr_pre_max": pre_max,
    })

peaks_df = pd.DataFrame(records).set_index("cycle_peak_year").sort_index()
# 选用“窗口峰”作为代表值（更鲁棒）
y_obs = peaks_df["ahr_peak_window_max"].copy()

# ============ 拟合“每轮衰减”表达式 ============
# 将 2017,2021,2025 映射到 n=0,1,2
years = y_obs.index.to_list()
n = np.arange(len(years))  # 0,1,2
y = y_obs.values.astype(float)

# 1) 自由拟合（log2 线性）：log2(y) = a + b*n  → 每轮倍率 f = 2^b
log2y = np.log2(y)
A = np.vstack([np.ones_like(n), n]).T
coef, *_ = np.linalg.lstsq(A, log2y, rcond=None)
a_hat, b_hat = coef[0], coef[1]
f_hat = 2.0 ** b_hat
y_fit_free = 2.0 ** (a_hat + b_hat * n)

# 2) 三个候选固定倍率：f ∈ {1/2, 1/√2, 1/4}
candidates = {
    "half(×0.5/轮)": 0.5,
    "sqrt2(×0.707/轮)": 1/np.sqrt(2),
    "quarter(×0.25/轮)": 0.25,
}

def best_c_for_factor(f, y):
    # 令 y_n ≈ c * f^n，求最小二乘 c*
    w = f ** n
    c = (y @ w) / (w @ w)
    return c, c * w

model_rows = []
for name, f in candidates.items():
    c_star, y_fit = best_c_for_factor(f, y)
    rmse_abs = float(np.sqrt(np.mean((y - y_fit) ** 2)))
    rmse_log2 = float(np.sqrt(np.mean((np.log2(y) - np.log2(y_fit)) ** 2)))
    model_rows.append({"model": name, "factor_per_cycle": f, "c_star": c_star,
                       "rmse_abs": rmse_abs, "rmse_log2": rmse_log2})

# 自由拟合的误差（对照）
rmse_abs_free  = float(np.sqrt(np.mean((y - y_fit_free) ** 2)))
rmse_log2_free = float(np.sqrt(np.mean((np.log2(y) - np.log2(y_fit_free)) ** 2)))
model_rows.append({"model": "free_fit(2^(a+b*n))", "factor_per_cycle": f_hat, "c_star": 2**a_hat,
                   "rmse_abs": rmse_abs_free, "rmse_log2": rmse_log2_free})

models_df = pd.DataFrame(model_rows).set_index("model").sort_values("rmse_log2")

# ============ 导出 Excel ============
with pd.ExcelWriter(XLSX_OUT, engine="openpyxl") as w:
    ahr.to_excel(w, sheet_name="ahr999_daily")
    peaks_df.to_excel(w, sheet_name="cycle_peaks")
    models_df.to_excel(w, sheet_name="decay_fits")
print("已保存分析表：", XLSX_OUT.resolve())

# ============ 画图：观测 vs 拟合 ============
plt.figure(figsize=(8,4.5))
plt.plot(n, y, marker="o", linestyle="-", label="观测（窗口峰 AHR999）")
# 画自由拟合
plt.plot(n, y_fit_free, marker="o", linestyle="--", label=f"自由拟合：×{f_hat:.3f}/轮")
# 画两个最优的候选模型（按 log2 RMSE 排名前2）
top2 = models_df.drop(index=["free_fit(2^(a+b*n))"], errors="ignore").head(2)
for mdl in top2.itertuples():
    f = mdl.factor_per_cycle
    c = mdl.c_star
    y_fit = c * (f ** n)
    plt.plot(n, y_fit, marker="o", linestyle=":", label=f"{mdl.Index}（×{f:.3f}/轮）")
plt.xticks(n, years)
plt.xlabel("循环（按峰值年）"); plt.ylabel("AHR999（窗口峰）")
plt.title("AHR999 在峰值处的跨轮衰减：观测 vs 拟合")
plt.legend()
plt.tight_layout()
plt.savefig(PLOT_FILENAME, dpi=170); plt.close()
print("已保存图：", PLOT_FILENAME.resolve())

# ============ 控制台简要汇总 ============
print("\n—— 峰值窗口内 AHR999 ——")
print(peaks_df[["ahr_peak_window_max", "ahr_peak_window_max_date"]])
print("\n—— 衰减拟合（按 log2 误差排序）——")
print(models_df)
print(f"\n自由拟合每轮倍率 f_hat = {f_hat:.4f}（与 0.5 的差 {abs(f_hat-0.5):.4f}；与 1/√2 的差 {abs(f_hat-1/np.sqrt(2)):.4f}）")


FileNotFoundError: 未找到 ahr999_daily.xlsx / ahr999_daily.csv，请先运行前面的 AHR999 生成脚本。

In [ ]:
from pathlib import Path
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from matplotlib import rcParams, font_manager
import matplotlib, shutil

# ============ 中文字体 & 负号修复（沿用你的做法） ============
print(matplotlib.get_cachedir())  # 如遇中文不生效，可手动删除该目录下 fontlist*.json

candidates = [
    r"C:\Windows\Fonts\msyh.ttc",     # 微软雅黑
    r"C:\Windows\Fonts\simhei.ttf",   # 黑体
    r"C:\Windows\Fonts\msyhbd.ttc",
]
for p in candidates:
    if Path(p).exists():
        font_manager.fontManager.addfont(p)

rcParams["font.family"] = "sans-serif"
rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "DejaVu Sans"]
rcParams["axes.unicode_minus"] = False

# ============ 文件与输出目录 ============
F15 = Path("btc_2015.xlsx")  # 需包含 ts, price 两列（或两行转置），price=USD/BTC
F21 = Path("btc_2021.xlsx")
F25 = Path("btc_2025.xlsx")
OUTDIR = Path("btc_fit_outputs"); OUTDIR.mkdir(exist_ok=True)

# ============ 工具函数（复用/精简） ============
def read_two_col_excel(fp: Path) -> pd.DataFrame:
    """读两列（时间、价格），兼容“横着两行”，去重、按时间升序。price 以 USD/BTC 为单位"""
    assert fp.exists(), f"未找到文件：{fp}"
    df_raw = pd.read_excel(fp, engine="openpyxl", header=None)
    if df_raw.shape[0] <= 3 and df_raw.shape[1] > df_raw.shape[0]:
        df_raw = df_raw.T
    df = df_raw.iloc[:, :2].copy()
    df.columns = ["ts", "price"]
    df["ts"] = pd.to_datetime(df["ts"], errors="coerce")
    df["price"] = pd.to_numeric(df["price"], errors="coerce")
    df = (df.dropna(subset=["ts","price"])
            .sort_values("ts")
            .drop_duplicates(subset=["ts"], keep="last"))
    return df

def to_daily_close(df: pd.DataFrame) -> pd.Series:
    """转日频收盘 (last) + 前向填充"""
    s = df.set_index("ts")["price"].resample("D").last()
    full_idx = pd.date_range(s.index.min().normalize(), s.index.max().normalize(), freq="D")
    s = s.reindex(full_idx).ffill()
    s.index.name = "ts"
    s.name = "usd_per_btc"
    return s

def assemble_price_series(files):
    frames = [read_two_col_excel(f) for f in files if f.exists()]
    assert frames, "没有可用的价格文件（检查 F15/F21/F25 是否存在）"
    df = pd.concat(frames, ignore_index=True)
    df = df.sort_values("ts").drop_duplicates("ts", keep="last")
    return to_daily_close(df)

# ============ 读数并构造两种“倒过来”的表示 ============
s = assemble_price_series([F15, F21, F25])     # s：USD/BTC（日频）
usd_per_btc = s

# A) 每聪价格（USD/sat），用于“倒过来 + 反向 Y 轴”的图（与示例外观一致）
usd_per_sat = usd_per_btc / 1e8
usd_per_sat.name = "usd_per_sat"

# B) BTC per USD（1 美元能买多少 BTC），可选图
btc_per_usd = 1.0 / usd_per_btc
btc_per_usd.name = "btc_per_usd"

# ============ 图1：示例同款（USD/sat，对数坐标 + Y 轴反向） ============
fig, ax = plt.subplots(figsize=(12, 4.5))
ax.plot(usd_per_sat.index, usd_per_sat.values, lw=1.2)
ax.set_yscale("log")
ax.invert_yaxis()  # 关键：把 Y 轴倒过来（数值从 1e-10 在上 到 1e-3 在下）
ax.set_xlabel("年份")
ax.set_ylabel("每聪价格 (USD/sat)")
ax.set_title("把美元计价的 BTC 倒过来：每聪价格（对数坐标，Y 轴反向）")
ax.grid(True, which="both", ls=":", alpha=0.35)

# 右侧加一个同步刻度，显示 USD/BTC，方便直觉理解
def sat_to_btc(y_sat):   # 左轴 → 右轴
    return y_sat * 1e8
def btc_to_sat(y_btc):   # 右轴 → 左轴
    return y_btc / 1e8

sec = ax.secondary_yaxis('right', functions=(sat_to_btc, btc_to_sat))
sec.set_ylabel("每枚价格 (USD/BTC)")

plt.tight_layout()
plt.savefig(OUTDIR / "btc_inverted_usd_per_sat.png", dpi=170)
plt.close()

# ============ 图2（可选）：BTC per USD（不反向，正常对数下降） ============
fig, ax = plt.subplots(figsize=(12, 4.5))
ax.plot(btc_per_usd.index, btc_per_usd.values, lw=1.2)
ax.set_yscale("log")
ax.set_xlabel("年份")
ax.set_ylabel("BTC / USD")
ax.set_title("1 美元能买多少 BTC（对数坐标）")
ax.grid(True, which="both", ls=":", alpha=0.35)
plt.tight_layout()
plt.savefig(OUTDIR / "btc_per_usd.png", dpi=170)
plt.close()

print("图已输出到：", OUTDIR.resolve())


C:\Users\nickc\.matplotlib


C:\Users\nickc\AppData\Local\Temp\ipykernel_27352\3248849729.py:38: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["ts"] = pd.to_datetime(df["ts"], errors="coerce")
C:\Users\nickc\AppData\Local\Temp\ipykernel_27352\3248849729.py:38: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["ts"] = pd.to_datetime(df["ts"], errors="coerce")


图已输出到： C:\Users\nickc\yanda\btc\btc_fit_outputs


In [ ]:
# -*- coding: utf-8 -*-
from pathlib import Path
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from matplotlib import rcParams, font_manager
import matplotlib

# ========= 中文字体（可选） =========
candidates = [
    r"C:\Windows\Fonts\msyh.ttc",   # 微软雅黑
    r"C:\Windows\Fonts\simhei.ttf", # 黑体
]
for p in candidates:
    if Path(p).exists(): font_manager.fontManager.addfont(p)
rcParams["font.family"] = "sans-serif"
rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "DejaVu Sans"]
rcParams["axes.unicode_minus"] = False

# ========= 输入 / 输出 =========
F15 = Path("btc_2015.xlsx")   # 两列：ts, price(USD/BTC)
F21 = Path("btc_2021.xlsx")
F25 = Path("btc_2025.xlsx")
OUTDIR = Path("btc_fit_outputs"); OUTDIR.mkdir(exist_ok=True)

GENESIS = pd.Timestamp("2009-01-03")      # 比特币“生日”
FIT_START = None                          # 可限定拟合起始日，例如 pd.Timestamp("2010-07-01")
FIT_END   = None                          # 可限定拟合结束日
QTRIM = 0.00                              # 量化截尾去极值（比如 0.01 表示去掉前后各1%）

# ========= 工具函数 =========
def read_two_col_excel(fp: Path) -> pd.DataFrame:
    assert fp.exists(), f"未找到文件：{fp}"
    df_raw = pd.read_excel(fp, engine="openpyxl", header=None)
    if df_raw.shape[0] <= 3 and df_raw.shape[1] > df_raw.shape[0]:
        df_raw = df_raw.T
    df = df_raw.iloc[:, :2].copy()
    df.columns = ["ts", "price"]
    df["ts"] = pd.to_datetime(df["ts"], errors="coerce")
    df["price"] = pd.to_numeric(df["price"], errors="coerce")
    return (df.dropna(subset=["ts","price"])
              .sort_values("ts")
              .drop_duplicates("ts", keep="last"))

def to_daily_close(df: pd.DataFrame) -> pd.Series:
    s = df.set_index("ts")["price"].resample("D").last()
    s = s.reindex(pd.date_range(s.index.min().normalize(), s.index.max().normalize(), freq="D")).ffill()
    s.index.name = "ts"; s.name = "usd_per_btc"
    return s

def assemble_price_series(files):
    frames = [read_two_col_excel(f) for f in files if f.exists()]
    assert frames, "没有可用的价格文件"
    df = pd.concat(frames, ignore_index=True).sort_values("ts").drop_duplicates("ts", keep="last")
    return to_daily_close(df)

def fit_loglog_age_price(usd_per_btc: pd.Series,
                         genesis=GENESIS,
                         start: pd.Timestamp|None=None,
                         end: pd.Timestamp|None=None,
                         qtrim: float=0.0):
    s = usd_per_btc.copy()
    if start: s = s[s.index >= pd.Timestamp(start)]
    if end:   s = s[s.index <= pd.Timestamp(end)]
    # 币龄（天）
    age_days = (s.index - pd.Timestamp(genesis)).days.astype(float)
    m = (age_days > 0) & (s.values > 0)
    X = np.log10(age_days[m])
    Y = np.log10(s.values[m])

    # 可选：量化截尾，减少极端点影响
    if qtrim > 0:
        loX, hiX = np.quantile(X, [qtrim, 1-qtrim])
        loY, hiY = np.quantile(Y, [qtrim, 1-qtrim])
        keep = (X>=loX)&(X<=hiX)&(Y>=loY)&(Y<=hiY)
        X, Y = X[keep], Y[keep]

    # 线性回归（log-log 空间）
    a, b = np.polyfit(X, Y, 1)     # Y = a*X + b
    # 拟合优度
    Yhat = a*X + b
    ss_res = np.sum((Y - Yhat)**2)
    ss_tot = np.sum((Y - Y.mean())**2)
    r2 = 1 - ss_res/ss_tot

    # 生成“拟合线”在原始时间序列上的值
    X_full = np.log10(age_days[m])              # 与 Y 同长度
    Y_fit_full = a*X_full + b
    fit_price = pd.Series(10**Y_fit_full, index=s.index[m], name="fit_usd_per_btc")

    stats = {"slope_a": a, "intercept_b": b, "r2": r2, "n": int(X.size)}
    return fit_price, stats

# ========= 读取并拟合 =========
s_usd = assemble_price_series([F15, F21, F25])           # USD/BTC（日频）
fit_line, stats = fit_loglog_age_price(s_usd, GENESIS, FIT_START, FIT_END, QTRIM)

# ========= 画图：和示例一致（双对数坐标） =========
fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(s_usd.index, s_usd.values, color="red", lw=1.2, label="比特币价格")
ax.plot(fit_line.index, fit_line.values, ls="--", lw=1.6, color="C0", label="币价指数拟合线")

# 横轴：币龄（天）的对数坐标（用第二个横轴来显示“币龄（天）”，主轴仍用日期以便画曲线连续）
# 更接近原图做法：把 x 轴直接替换为“币龄（天）”
age_days_full = (s_usd.index - GENESIS).days.astype(float)
ax.set_xscale("log")
ax.set_xlim(age_days_full.min() if age_days_full.min()>0 else 1, age_days_full.max())
# 用币龄刻度画，但曲线仍按时间顺序连接（视觉与原图一致）
ax.set_xlabel("比特币天龄（天）")

# 把 x 数据映射到“币龄（天）”
# 注意：Matplotlib 允许我们只设置刻度标签为天龄，这里直接改 ticks/labels
# 为了简单，这里让 Matplotlib 自动放置对数刻度；需要自定义可用 LogLocator

# 纵轴：价格（美元）的对数坐标
ax.set_yscale("log")
ax.set_ylabel("比特币价格（美元）")
ax.grid(True, which="both", ls=":", alpha=0.35)

# 在图中标注公式 Y=aX+b（X=log10(币龄), Y=log10(价格)）
a, b, r2 = stats["slope_a"], stats["intercept_b"], stats["r2"]
text = f"Y = {a:.2f}X {b:+.2f}\n(对数空间)  R² = {r2:.3f}"
# 选一个合适的位置
ax.text(0.55, 0.15, text, transform=ax.transAxes, fontsize=11,
        bbox=dict(boxstyle="round,pad=0.35", fc="white", ec="gray", alpha=0.8))

ax.legend()
ax.set_title("币价 vs 币龄（双对数拟合）")

plt.tight_layout()
plt.savefig(OUTDIR / "btc_price_vs_age_loglog_fit.png", dpi=170)
plt.close()

# ========= 保存参数 =========
with open(OUTDIR / "btc_price_vs_age_fit_stats.txt","w",encoding="utf-8") as f:
    f.write(f"模型：log10(价格) = a * log10(币龄天) + b\n")
    f.write(f"a(斜率) = {a:.6f}\n")
    f.write(f"b(截距) = {b:.6f}\n")
    f.write(f"R^2    = {r2:.6f}\n")
    f.write(f"N      = {stats['n']}\n")
    f.write(f"等价：价格 ≈ 10^b * (币龄天)^a\n")
print("完成：", (OUTDIR / "btc_price_vs_age_loglog_fit.png").resolve())


# ========= 画图（修正版）：x 用“币龄（天）”而不是日期 =========
from matplotlib.ticker import LogLocator

# 1) 原始价格曲线转为 (age_days, price)
age_days_full = (s_usd.index - GENESIS).days.astype(float)
mask = (age_days_full > 0) & (s_usd.values > 0)
x_age = age_days_full[mask]
y_price = s_usd.values[mask]

# 2) 拟合线同样用币龄（天）做 x
x_fit = (fit_line.index - GENESIS).days.astype(float)
y_fit = fit_line.values

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(x_age, y_price, color="red", lw=1.2, label="比特币价格")
ax.plot(x_fit, y_fit, ls="--", lw=1.6, color="C0", label="币价指数拟合线")

# 对数-对数坐标
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlim(x_age.min(), x_age.max())

# 美化：网格 & 轴标签
ax.grid(True, which="both", ls=":", alpha=0.35)
ax.set_xlabel("比特币天龄（天）")
ax.set_ylabel("比特币价格（美元）")
ax.set_title("币价 vs 币龄（双对数拟合）")

# 在图中标注拟合公式（X=log10(币龄), Y=log10(价格)）
a, b, r2 = stats["slope_a"], stats["intercept_b"], stats["r2"]
ax.text(0.55, 0.15, f"Y = {a:.2f}X {b:+.2f}\n(对数空间)  R² = {r2:.3f}",
        transform=ax.transAxes, fontsize=11,
        bbox=dict(boxstyle="round,pad=0.35", fc="white", ec="gray", alpha=0.8))

ax.legend()
plt.tight_layout()
plt.savefig(OUTDIR / "btc_price_vs_age_loglog_fit.png", dpi=170)
plt.close()




C:\Users\nickc\AppData\Local\Temp\ipykernel_27352\2486781315.py:38: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["ts"] = pd.to_datetime(df["ts"], errors="coerce")
C:\Users\nickc\AppData\Local\Temp\ipykernel_27352\2486781315.py:38: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["ts"] = pd.to_datetime(df["ts"], errors="coerce")


完成： C:\Users\nickc\yanda\btc\btc_fit_outputs\btc_price_vs_age_loglog_fit.png


In [6]:
# -*- coding: utf-8 -*-
from pathlib import Path
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from matplotlib import rcParams, font_manager

# ========= 中文字体（可选） =========
for p in [r"C:\Windows\Fonts\msyh.ttc", r"C:\Windows\Fonts\simhei.ttf"]:
    if Path(p).exists(): font_manager.fontManager.addfont(p)
rcParams["font.family"] = "sans-serif"
rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "DejaVu Sans"]
rcParams["axes.unicode_minus"] = False

# ========= 输入 / 输出 =========
F15 = Path("btc_2015.xlsx")   # 两列：ts, price(USD/BTC)
F21 = Path("btc_2021.xlsx")
F25 = Path("btc_2025.xlsx")
OUTDIR = Path("btc_fit_outputs09142025"); OUTDIR.mkdir(exist_ok=True)

GENESIS = pd.Timestamp("2009-01-03")
QTRIM   = 0.00    # 截尾比例，用于稳健回归（0~0.05）
VOL_WIN = 120     # 滚动窗口天数（残差波动）
EMA_SPAN= 30      # EMA 平滑 span
K_CLIP  = (0.6, 3.0)  # 放大系数的剪切范围，避免失真
MONOTONE_INC = True   # 放大系数是否做 cummax（单调不减）
TARGET_Q = 0.80       # 目标波动的分位数（基于历史 sigma_t）

# ========= IO & 拟合 =========
def read_two_col_excel(fp: Path) -> pd.DataFrame:
    assert fp.exists(), f"未找到文件：{fp}"
    df_raw = pd.read_excel(fp, engine="openpyxl", header=None)
    if df_raw.shape[0] <= 3 and df_raw.shape[1] > df_raw.shape[0]:
        df_raw = df_raw.T
    df = df_raw.iloc[:, :2].copy()
    df.columns = ["ts", "price"]
    df["ts"] = pd.to_datetime(df["ts"], errors="coerce")
    df["price"] = pd.to_numeric(df["price"], errors="coerce")
    return (df.dropna(subset=["ts","price"])
              .sort_values("ts")
              .drop_duplicates("ts", keep="last"))

def to_daily_close(df: pd.DataFrame) -> pd.Series:
    s = df.set_index("ts")["price"].resample("D").last()
    s = s.reindex(pd.date_range(s.index.min().normalize(), s.index.max().normalize(), freq="D")).ffill()
    s.index.name = "ts"; s.name = "usd_per_btc"
    return s

def assemble_price_series(files):
    frames = [read_two_col_excel(f) for f in files if f.exists()]
    assert frames, "没有可用的价格文件"
    df = pd.concat(frames, ignore_index=True).sort_values("ts").drop_duplicates("ts", keep="last")
    return to_daily_close(df)

def fit_loglog_age_price(usd_per_btc: pd.Series, genesis=GENESIS, qtrim=0.0):
    # 幣齡（天）→ float ndarray（避免 Index 类型）
    age_days = ((usd_per_btc.index.values.astype('datetime64[ns]') -
                 np.datetime64(genesis)) / np.timedelta64(1, 'D')).astype(float)
    px = usd_per_btc.values.astype(float)

    m = (age_days > 0) & (px > 0)
    X = np.log10(age_days[m]).astype(float)   # ndarray
    Y = np.log10(px[m]).astype(float)         # ndarray

    if qtrim and qtrim > 0:
        loX, hiX = np.quantile(X, [qtrim, 1-qtrim])
        loY, hiY = np.quantile(Y, [qtrim, 1-qtrim])
        keep = (X >= loX) & (X <= hiX) & (Y >= loY) & (Y <= hiY)
        X, Y = X[keep], Y[keep]

    # 线性回归（log-log）
    a, b = np.polyfit(X, Y, 1)
    Yhat = a * X + b

    # R^2（显式用 numpy）
    ss_res = np.sum((Y - Yhat) ** 2)
    ss_tot = np.sum((Y - np.mean(Y)) ** 2)
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan
    return float(a), float(b), float(r2)


# ========= 等波动放大器 =========
def equalize_volatility(usd_per_btc: pd.Series, a: float, b: float,
                        genesis=GENESIS, vol_win=120, ema_span=30,
                        target_q=0.8, k_clip=(0.6, 3.0), monotone_inc=True):
    idx = usd_per_btc.index
    age = (idx - genesis).days.astype(float)
    m = (age > 0) & (usd_per_btc.values > 0)

    x = pd.Series(np.log10(age[m]), index=idx[m])
    y = pd.Series(np.log10(usd_per_btc.values[m]), index=idx[m])

    y_trend = a*x + b
    resid   = y - y_trend                 # 对数空间残差

    # 1) 滚动波动
    sigma = resid.rolling(vol_win, min_periods=max(15, vol_win//3)).std(ddof=0)
    sigma = sigma.bfill().ffill()

    # 2) 目标波动（分位数/中值/早期区间皆可，这里用分位数）
    sigma_target = float(sigma.quantile(target_q))

    # 3) 放大系数 k_t = sigma_target / sigma_t
    k = sigma_target / sigma
    if ema_span and ema_span > 1:
        k = k.ewm(span=ema_span, adjust=False).mean()
    if monotone_inc:
        k = k.cummax()                    # 单调递增
    k = k.clip(lower=k_clip[0], upper=k_clip[1])

    # 4) 应用在对数空间（围绕趋势放大残差）
    y_adj = y_trend + k * resid
    p_adj = pd.Series(10**y_adj, index=y_adj.index, name="price_equalized")

    # 打包返回
    out = pd.DataFrame({
        "usd_per_btc": usd_per_btc.reindex(p_adj.index),
        "price_equalized": p_adj,
        "residual_log10": resid,
        "sigma_roll": sigma,
        "k_equalizer": k,
    })
    return out, sigma_target

# ========= 主流程 =========
s_usd = assemble_price_series([F15, F21, F25])
a, b, r2 = fit_loglog_age_price(s_usd, GENESIS, QTRIM)
df_eq, sigma_star = equalize_volatility(
    s_usd, a, b, GENESIS,
    vol_win=VOL_WIN, ema_span=EMA_SPAN,
    target_q=TARGET_Q, k_clip=K_CLIP, monotone_inc=MONOTONE_INC
)

# ========= 输出 CSV =========
df_eq.to_csv(OUTDIR / "btc_equalized_series.csv", encoding="utf-8-sig")

# ========= 图1：币龄-对数坐标中，原价/拟合线/等波动价 =========
import numpy as np
age_days = (df_eq.index - GENESIS).days.astype(float)
x_age = age_days[age_days > 0]

# 拟合线（画在相同 x 上）
x_log = np.log10(x_age)
fit_line = 10**(a*x_log + b)

plt.figure(figsize=(14, 6.5))
# 原价
plt.plot(x_age, df_eq.loc[df_eq.index[age_days>0], "usd_per_btc"].values,
         color="red", lw=1.0, label="比特币价格（原始）")
# 等波动价格
plt.plot(x_age, df_eq.loc[df_eq.index[age_days>0], "price_equalized"].values,
         color="C2", lw=1.2, alpha=0.9, label="等波动价格（放大残差）")
# 拟合线
plt.plot(x_age, fit_line, "C0--", lw=1.6, label="币价指数拟合线")

plt.xscale("log"); plt.yscale("log")
plt.xlabel("比特币天龄（天）"); plt.ylabel("比特币价格（美元）")
plt.title("币价 vs 币龄（等波动校正后，同尺度对比）")
plt.grid(True, which="both", ls=":", alpha=0.35)
plt.legend()
plt.tight_layout()
plt.savefig(OUTDIR / "btc_price_equalized_vs_age.png", dpi=170)
plt.close()

# ========= 图2：放大系数 k_t 与滚动波动 σ_t =========
fig, ax1 = plt.subplots(figsize=(14, 5.2))
t = df_eq.index
ax1.plot(t, df_eq["k_equalizer"], lw=1.2, label="k_t 放大系数")
ax1.set_ylabel("k_t（单调递增）"); ax1.grid(True, ls=":", alpha=0.3)
ax1.legend(loc="upper left")

ax2 = ax1.twinx()
ax2.plot(t, df_eq["sigma_roll"], "C3--", lw=1.0, label="σ_t 残差滚动波动")
ax2.set_ylabel("σ_t（log10残差 std）")
ax2.legend(loc="upper right")
plt.title(f"等波动放大器：目标σ* = {sigma_star:.4f}，窗口={VOL_WIN}天，EMA={EMA_SPAN}")
plt.tight_layout()
plt.savefig(OUTDIR / "btc_equalizer_k_sigma.png", dpi=170)
plt.close()

# ========= 控制台提示 =========
print(f"a={a:.4f}, b={b:.4f}, R^2={r2:.3f} | 目标σ*={sigma_star:.4f}")
print("输出：", OUTDIR.resolve())


C:\Users\nickc\AppData\Local\Temp\ipykernel_17908\1332515266.py:36: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["ts"] = pd.to_datetime(df["ts"], errors="coerce")
C:\Users\nickc\AppData\Local\Temp\ipykernel_17908\1332515266.py:36: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["ts"] = pd.to_datetime(df["ts"], errors="coerce")


a=5.9474, b=-17.4588, R^2=0.885 | 目标σ*=0.0945
输出： C:\Users\nickc\yanda\btc\btc_fit_outputs09142025
